UPSC essay feedback analyzer: parallel workflow to create 3 feedbacks basis of COT, DOA, Language giving score and text, output: summarize the feedback, and average score 

In [12]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

import os
load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY")

In [3]:
model = ChatOpenAI(model='gpt-4o-mini')


In [4]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback on the essay")
    score: int = Field(description="Score out of 10 for the essay", ge=0, le=10)

In [6]:
structured_model = model.with_structured_output(EvaluationSchema)

In [8]:
essay = "India is rapidly emerging as a global powerhouse in Artificial Intelligence, prioritizing \"AI for Public Good\" to drive socio-economic transformation. By leveraging its massive talent pool, a billion-plus digital infrastructure, and targeted state policies, India is localizing AI to solve grassroots challenges in agriculture, healthcare, and governance while boosting overall economic growth.1. Harnessing Digital Public Infrastructure (DPI)India’s success in AI is deeply rooted in its foundational digital systems like Aadhaar, UPI, and the Open Network for Digital Commerce (ONDC). By integrating AI with these population-scale platforms, the country is delivering inclusive governance, targeted financial inclusion, and transparent public services to its citizens.2. The IndiaAI MissionTo cement its position as a global AI leader, the Government of India launched the ambitious IndiaAI Mission and Emerging AI Ecosystem, committing over ₹10,300 crore toward building sovereign AI capabilities. Key initiatives include:Compute Access: Deploying 38,000+ GPUs to provide affordable, high-performance computing to startups and researchers.Indigenous Models: Funding localized foundation models, such as BharatGen, specifically trained to cater to India's diverse languages and cultural contexts.3. Transforming Core SectorsAI is reshaping fundamental pillars of the Indian economy:Healthcare: AI-enabled telemedicine and early-diagnostic tools are bridging the gap between urban specialists and rural populations.Agriculture: AI platforms analyze satellite imagery and weather data to provide precise advisory on crop health and irrigation, protecting smallholder farmers from climate variables.Language Translation: Platforms like BHASHINI break down linguistic barriers, enabling citizens to access digital services and government schemes in their regional languages.4. Global Talent and Enterprise AdoptionIndia possesses the third-largest pool of AI talent globally and houses over 500 Global Capability Centres (GCCs) focused strictly on artificial intelligence. Furthermore, Indian enterprises are transitioning from AI experimentation to full-scale deployment, with sectors such as banking, finance, and consumer retail leading the charge. The country's software services industry is projected to play a massive role in building and deploying these tools on a global scale.5. Responsible AI and GovernanceAs AI scales, India emphasizes safe and ethical deployment. The country has introduced principle-based governance frameworks, such as the India AI Governance Guidelines, to prioritize fairness, accountability, and the mitigation of biases in AI models. ConclusionIndia’s role in AI represents a unique intersection of rapid technological innovation and inclusive development. By focusing on sovereign capabilities and localized solutions, India is building an ecosystem that not only fuels economic growth but also ensures that the benefits of AI reach the most marginalized communities."

In [10]:
prompt = f"Evaluate the language quality of the following essay on a scale of 0 to 10, providing detailed feedback. The essay is: {essay}"
structured_model.invoke(prompt).score

8

In [ ]:
class UPSCState(TypedDict):
    essay: str
    COT_feedback: str
    DOA_feedback: str
    Language_feedback: str
    individual_scores: Annotated[list[int],operator.add] #reducer function to merge individual scores
    final_score: float
    final_feedback: str

In [26]:
def COT_feedback(state: UPSCState) -> dict:
    prompt = f"Provide detailed feedback on the essay's clarity of thought and assign a score out of 10. The essay is: {state['essay']}"
    output = structured_model.invoke(prompt)
    return {"COT_feedback": output.feedback, "individual_scores": [output.score]}

In [27]:
def DOA_feedback(state: UPSCState) -> dict:
    prompt = f"Provide detailed feedback on the essay's Depth of analysis and assign a score out of 10. The essay is: {state['essay']}"
    output = structured_model.invoke(prompt)
    return {"DOA_feedback": output.feedback, "individual_scores": [output.score]}

In [28]:
def Language_feedback(state: UPSCState) -> dict:
    prompt = f"Provide detailed feedback on the essay's language and assign a score out of 10. The essay is: {state['essay']}"
    output = structured_model.invoke(prompt)
    return {"Language_feedback": output.feedback, "individual_scores": [output.score]}

In [31]:
def final_feedback(state: UPSCState) -> dict:
    #summary feedback and final score
    prompt = f" Based on the previous feedback provided by DOA, COT and Language feedback, provide a final summary feedback on the essay. \n The COT feedback essay is: {state['COT_feedback']}\n The DOA feedback essay is: {state['DOA_feedback']}\n The Language feedback essay is: {state['Language_feedback']}"
    feedback_summary = structured_model.invoke(prompt)
    final_score = sum(state['individual_scores'])/len(state['individual_scores'])  # Calculate the average score from the individual scores
    return {"final_score": final_score, "final_feedback": feedback_summary}

In [33]:
#create a graph
graph = StateGraph(UPSCState)

#add nodes
graph.add_node("COT_feedback", COT_feedback)
graph.add_node("DOA_feedback", DOA_feedback)
graph.add_node("Language_feedback", Language_feedback)   
graph.add_node("final_feedback", final_feedback)

#add edges
graph.add_edge(START, "COT_feedback")
graph.add_edge(START, "DOA_feedback")
graph.add_edge(START, "Language_feedback")
graph.add_edge("COT_feedback", "final_feedback")
graph.add_edge("DOA_feedback", "final_feedback")
graph.add_edge("Language_feedback", "final_feedback")  
graph.add_edge("final_feedback", END) 

workflow = graph.compile()
initial_state = {"essay": essay}
final_state = workflow.invoke(initial_state)
final_state

{'essay': 'India is rapidly emerging as a global powerhouse in Artificial Intelligence, prioritizing "AI for Public Good" to drive socio-economic transformation. By leveraging its massive talent pool, a billion-plus digital infrastructure, and targeted state policies, India is localizing AI to solve grassroots challenges in agriculture, healthcare, and governance while boosting overall economic growth.1. Harnessing Digital Public Infrastructure (DPI)India’s success in AI is deeply rooted in its foundational digital systems like Aadhaar, UPI, and the Open Network for Digital Commerce (ONDC). By integrating AI with these population-scale platforms, the country is delivering inclusive governance, targeted financial inclusion, and transparent public services to its citizens.2. The IndiaAI MissionTo cement its position as a global AI leader, the Government of India launched the ambitious IndiaAI Mission and Emerging AI Ecosystem, committing over ₹10,300 crore toward building sovereign AI ca